# WGCNA
## 1.简介  
WGCNA（Weighted Gene Co-expression Network Analysis，加权基因共表达网络分析）是一种用于分析高维基因表达数据的系统生物学方法。它通过构建基因共表达网络来发现基因模块，并研究这些模块与外部表型之间的关联。WGCNA常用于RNA-seq、微阵列数据等高通量基因表达数据的分析，目标是识别与生物学过程或疾病相关的重要基因模块和核心基因。

## 2.WGCNA的主要特点  
1) 加权网络：WGCNA构建的是加权网络，其中基因之间的相关性不简单地用0和1表示，而是根据它们的相关性程度进行加权。这种方法比传统的未加权网络更加稳定，并且能捕捉到基因表达间的渐进式关系。

2) 模块化分析：基因被聚类为表达模式相似的模块。每个模块由多个在不同样本中表现出共表达关系的基因组成，通常代表某种生物学功能或过程。模块中的基因被认为是共同参与某个生物学功能的可能候选者。

3) 模块与表型的关联：WGCNA不仅识别了基因共表达模块，还可以将这些模块与外部表型（如疾病状态、临床特征等）进行关联分析，揭示特定模块的功能或生物学意义。

4) 核心基因（Hub genes）：在每个模块中，WGCNA还可以识别核心基因，这些基因与其他基因的连接度较高，可能在模块中的调控网络中扮演关键角色。

## 3.WGCNA的主要步骤  
1) 基因表达数据的预处理：输入数据为基因表达矩阵（行是基因，列是样本），对数据进行过滤和标准化，去除质量不佳的样本或基因。

2) 网络构建：计算基因间的相关性矩阵，选择一个合适的软阈值（power）来构建加权的基因共表达网络。这个步骤确保网络具有无标度特性，即少数基因有大量连接（hub genes），多数基因有较少连接。

3) 模块识别：基于拓扑重叠矩阵（TOM），通过层次聚类方法识别共表达基因模块。模块通常由颜色标签标记，如“蓝色模块”“棕色模块”。

4) 模块与性状的关联分析：使用模块特征基因（Module Eigengene, ME，表示模块中所有基因的主成分）来关联基因模块与样本的外部性状（如疾病状态、临床特征），确定哪些模块可能与特定性状相关。

5) 识别核心基因：在与感兴趣的性状相关的模块中，进一步识别核心基因，这些基因在模块中有最高的连接度，可能是调控该模块的关键节点。





## `run_analysis.py` 使用说明

`run_analysis.py` 是一个用于分析基因表达数据的 Python 脚本，结合 R 语言的 WGCNA 方法，生成基因共表达网络和相关性分析结果。

### 运行方式

您可以通过命令行运行此脚本。运行方式如下：

```bash
python run_analysis.py --c < count_file > --p < pheno_file > --t < trait >--o < output_dir >

例如： python run_analysis.py --c test.txt --p pheno.txt --t "3D" --o "/mnt/kongtianci/CDesk/WGCNA/WGC
NA_results"

count_file：包含基因表达计数数据的文件路径，通常是一个.txt或.csv文件。
格式：第一列为基因ID，后续列为样本数据。每一行代表一个基因，每一列代表一个样本。
GeneID Sample1 Sample2 Sample3 ...
GeneA   10      15      20
GeneB   5       7       8

pheno_file：包含表型信息的文件路径，用于描述每个样本的特征或实验条件。
格式：文件必须包含至少两列：sample_id和condition。sample_id对应count_file文件中的样本列，condition则是样本所属的实验组或特征。
sample_id   condition
Sample1     Control
Sample2     Treated
Sample3     Treated

triat：表型信息，计算基因模块与表型的相关性。

output_dir：输出目录路径，用于保存分析结果。脚本会自动在指定目录下生成分析结果的文件，包括热图、模块性状相关性热图、密度图等。如果目录不存在，脚本会自动创建该目录。
'''


## 环境依赖
### Python 版本：3.8.19
### R 版本：4.2.3

必要的 Python 包：rpy2, pandas, matplotlib, seaborn

R 依赖：WGCNA, ggplot2, reshape, gplots, RColorBrewer



In [1]:
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import pandas as pd

In [2]:
pandas2ri.activate()#启用 pandas 和 R 之间的数据结构转换的函数

In [3]:
ro.r('.libPaths(c("/mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2", '
      '"/home/sugon/Install/R-4.2/lib/R/library"))')


In [4]:
ro.r('.libPaths(c("/mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2"))')#添加环境


In [12]:
ro.r('install.packages("gplots")')#安装gplots包

R[write to console]: Installing package into ‘/mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2’
(as ‘lib’ is unspecified)



--- Please select a CRAN mirror for use in this session ---
Secure CRAN mirrors 

 1: 0-Cloud [https]
 2: Australia (Canberra) [https]
 3: Australia (Melbourne 1) [https]
 4: Australia (Melbourne 2) [https]
 5: Austria (Wien 1) [https]
 6: Belgium (Brussels) [https]
 7: Brazil (PR) [https]
 8: Brazil (SP 1) [https]
 9: Brazil (SP 2) [https]
10: Bulgaria [https]
11: Canada (MB) [https]
12: Canada (ON 1) [https]
13: Canada (ON 2) [https]
14: Chile (Santiago) [https]
15: China (Beijing 2) [https]
16: China (Beijing 3) [https]
17: China (Hefei) [https]
18: China (Hong Kong) [https]
19: China (Guangzhou) [https]
20: China (Jinan) [https]
21: China (Nanjing) [https]
22: China (Shanghai 2) [https]
23: China (Shenzhen) [https]
24: China (Wuhan) [https]
25: Colombia (Cali) [https]
26: Costa Rica [https]
27: Cyprus [https]
28: Czech Republic [https]
29: Denmark [https]
30: East Asia [https]
31: Ecuador (Cuenca) [https]
32: France (Lyon 1) [https]
33: France (Lyon 2) [https]
34: France (Marseille

R[write to console]: also installing the dependencies ‘gtools’, ‘caTools’


R[write to console]: trying URL 'https://mirrors.ustc.edu.cn/CRAN/src/contrib/gtools_3.9.5.tar.gz'

R[write to console]: Content type 'application/octet-stream'
R[write to console]:  length 242029 bytes (236 KB)

R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: =
R[write to console]: 

x86_64-conda-linux-gnu-cc -I"/mnt/kongtianci/anaconda3/envs/myenv/lib/R/include" -DNDEBUG   -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /mnt/kongtianci/anaconda3/envs/myenv/include -I/mnt/kongtianci/anaconda3/envs/myenv/include -Wl,-rpath-link,/mnt/kongtianci/anaconda3/envs/myenv/lib    -fpic  -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /mnt/kongtianci/anaconda3/envs/myenv/include -fdebug-prefix-map=/home/conda/feedstock_root/build_artifacts/r-base-split_1720332503261/work=/usr/local/src/conda/r-base-4.3.3 -fdebug-prefix-map=/mnt/kongtianci/anaconda3/envs/myenv=/usr/local/src/conda-prefix  -c init.c -o init.o
x86_64-conda-linux-gnu-cc -I"/mnt/kongtianci/anaconda3/envs/myenv/lib/R/include" -DNDEBUG   -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /mnt/kongtianci/anaconda3/envs/myenv/include -I/mnt/kongtianci/anaconda3/envs/myenv/include -Wl,-rpath-link,/mnt/kongtianci/anaconda3/envs/myenv/lib    -fpic  -march=noco

installing to /mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2/00LOCK-gtools/00new/gtools/libs
** R
** data
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
*** copying figures
** building package indices
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (gtools)
* installing *source* package ‘caTools’ ...
** package ‘caTools’ successfully unpacked and MD5 sums checked
** using staged installation
** libs
using C compiler: ‘x86_64-conda-linux-gnu-cc (conda-forge gcc 14.1.0-0) 14.1.0’
using C++ compiler: ‘x86_64-conda-linux-gnu-c++ (conda-forge gcc 14.1.0-0) 14.1.0’


x86_64-conda-linux-gnu-c++ -std=gnu++17 -I"/mnt/kongtianci/anaconda3/envs/myenv/lib/R/include" -DNDEBUG   -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /mnt/kongtianci/anaconda3/envs/myenv/include -I/mnt/kongtianci/anaconda3/envs/myenv/include -Wl,-rpath-link,/mnt/kongtianci/anaconda3/envs/myenv/lib    -fpic  -fvisibility-inlines-hidden  -fmessage-length=0 -march=nocona -mtune=haswell -ftree-vectorize -fPIC -fstack-protector-strong -fno-plt -O2 -ffunction-sections -pipe -isystem /mnt/kongtianci/anaconda3/envs/myenv/include -fdebug-prefix-map=/home/conda/feedstock_root/build_artifacts/r-base-split_1720332503261/work=/usr/local/src/conda/r-base-4.3.3 -fdebug-prefix-map=/mnt/kongtianci/anaconda3/envs/myenv=/usr/local/src/conda-prefix  -c Gif2R.cpp -o Gif2R.o
x86_64-conda-linux-gnu-c++ -std=gnu++17 -I"/mnt/kongtianci/anaconda3/envs/myenv/lib/R/include" -DNDEBUG   -DNDEBUG -D_FORTIFY_SOURCE=2 -O2 -isystem /mnt/kongtianci/anaconda3/envs/myenv/include -I/mnt/kongtianci/anaconda3/envs/myenv/includ

installing to /mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2/00LOCK-caTools/00new/caTools/libs
** R
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** testing if installed package can be loaded from temporary location
** checking absolute paths in shared objects and dynamic libraries
** testing if installed package can be loaded from final location
** testing if installed package keeps a record of temporary installation path
* DONE (caTools)
* installing *source* package ‘gplots’ ...
** package ‘gplots’ successfully unpacked and MD5 sums checked
** using staged installation
** R
** data
*** moving datasets to lazyload DB
** inst
** byte-compile and prepare package for lazy loading
** help
*** installing help indices
** building package indices
** installing vignettes
** testing if installed package can be loaded from temporary location
** testing if installed package can be loaded from final location
** testing if 

In [6]:
# 尝试加载 WGCNA
try:
    ro.r('library(WGCNA)')
    print("WGCNA 加载成功！")
except Exception as e:
    print(f"加载 WGCNA 时出错：{e}")

R[write to console]: Loading required package: dynamicTreeCut

R[write to console]: Loading required package: fastcluster

R[write to console]: 
Attaching package: ‘fastcluster’


R[write to console]: The following object is masked from ‘package:stats’:

    hclust


R[write to console]: 

R[write to console]: Registered S3 methods overwritten by 'backports':
  method                    from 
  as.character.Rconcordance tools
  print.Rconcordance        tools

R[write to console]: 
Attaching package: ‘WGCNA’


R[write to console]: The following object is masked from ‘package:stats’:

    cor




WGCNA 加载成功！


In [31]:
print(ro.r('.libPaths()'))#查看R语言环境

['/mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2'
 '/mnt/kongtianci/anaconda3/envs/myenv/lib/R/library']


In [32]:
print(ro.r('system.file(package = "WGCNA")'))#查看WGCNA包安装的位置

['/mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2/WGCNA']


In [36]:
import os

os.environ["R_LIBS_USER"] = "/mnt/kongtianci/R/x86_64-pc-linux-gnu-library/4.2"

In [7]:
ro.r('source("analysis.R")')

In [8]:
analyze_expression = ro.globalenv['analyze_expression']

In [9]:
count = pd.read_csv('test.txt', sep='\t')  # 加载 count 数据
pheno = pd.read_csv('pheno.txt', sep='\t')    

In [10]:
r_count = pandas2ri.py2rpy(count)
r_pheno = pandas2ri.py2rpy(pheno)

In [11]:
output_dir = "/mnt/kongtianci/CDesk/WGCNA/WGCNA_results"